# CONDACT quick-start tutorial (Google Colab)

**CONDitional ACTivity: a kinetic framework for mapping allosteric communication**

Loverde Laboratory, CUNY. Code by Augustine C. Onyema, assisted by Chukwuebuka Dikeocha.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CUNY-CSI-Loverde-Laboratory/condact-colab-tutorial/blob/main/CONDACT_Colab_Tutorial.ipynb)

This notebook runs the whole CONDACT workflow on the small example trajectory that ships with the package, so it finishes in a few minutes on a free Colab runtime (no GPU needed).

**What you will do**

1. Install CONDACT
2. Load a trajectory with MDAnalysis
3. Look at the dihedral-angle distribution and choose the state boundaries
4. Compute the conditional activity matrix
5. Read off dynamical memory and plot the matrix
6. Test which couplings are statistically significant (permutation test + FDR)
7. Trace a communication route
8. Swap in your own trajectory

**The idea in one paragraph.** Each residue is reduced to one degree of freedom: the side-chain dihedral χ1 for amino acids (ALA and GLY are skipped), or the sugar-base dihedral for nucleotides. The angle is split into discrete states, and CONDACT records the times at which each residue switches state. For two residues X and Y,

$$A[X][Y] = -\ln\left(\frac{\langle\tau_{X|Y}\rangle}{\langle\tau_X\rangle}\right)$$

where $\langle\tau_X\rangle$ is the persistence time of X and $\langle\tau_{X|Y}\rangle$ is the mean time X waits to switch after Y has switched.

| Value | Meaning |
|---|---|
| `A[X][Y] > 0` | a transition in Y promotes a transition in X |
| `A[X][Y] = 0` | the two are independent |
| `A[X][Y] < 0` | a transition in Y suppresses a transition in X |
| `A[X][X]` | dynamical memory of residue X |

The matrix is directional: `A[X][Y]` is generally not equal to `A[Y][X]`.

## 1. Install CONDACT

This installs the package and its dependency MDAnalysis straight from GitHub. It takes about a minute.

In [ ]:
!pip install -q git+https://github.com/CUNY-CSI-Loverde-Laboratory/conditional_activity.git

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import MDAnalysis as mda

import conditional_activity
from conditional_activity.analysis.condact import CONDACT
from conditional_activity.analysis.fdr import CONDACT_FDR
from conditional_activity.analysis.trace_route import CONDACT_CommPath

print("CONDACT", conditional_activity.__version__, "| MDAnalysis", mda.__version__)

## 2. Load a trajectory

The package ships with a short example: a small peptide, saved every 10 ps. Any topology/trajectory pair that MDAnalysis can read will work the same way (see section 8).

Three settings are used throughout the notebook, so they are defined once here.

In [ ]:
# Example files installed with the package
example_dir = os.path.join(os.path.dirname(conditional_activity.__file__), "analysis")
topology_file = os.path.join(example_dir, "pytest_LYS.prmtop")
trajectory_file = os.path.join(example_dir, "pytest_LYS.xtc")

SELECTED_RESID = "1-3"   # residues to analyse, MDAnalysis resid syntax ("1-3", "5 9 12", ...)
SAVING_FREQUENCY = 10    # time between saved frames, in picoseconds (NOT the MD time step)

u = mda.Universe(topology_file, trajectory_file)
TOTAL_TIME = len(u.trajectory) * SAVING_FREQUENCY   # length of the simulation in ps

print(f"{u.atoms.n_atoms} atoms, {len(u.trajectory)} frames, {TOTAL_TIME/1000:.0f} ns")
print("Residues:", ", ".join(f"{r.resname} {r.resid}" for r in u.select_atoms(f"resid {SELECTED_RESID}").residues))

## 3. Choose the state boundaries

CONDACT needs to know how to cut the dihedral angle into states. Plot the distribution of χ1 over the trajectory, count the peaks, and put a boundary in each valley.

Angles are reported on 0–360° (negative angles have 360° added), which is what CONDACT uses by default.

In [ ]:
sel = u.select_atoms(f"resid {SELECTED_RESID} and not (resname ALA or resname GLY)")
chi1_groups = []
for res in sel.residues:
    # chi1 = N - CA - CB - (CG / OG1 / CG1 / OG / SG)
    ag = sum([res.atoms.select_atoms("name N"), res.atoms.select_atoms("name CA"),
              res.atoms.select_atoms("name CB"),
              res.atoms.select_atoms("name CG or name OG1 or name CG1 or name OG or name SG")])
    chi1_groups.append((f"{res.resname} {res.resid}", ag))

angles = {name: [] for name, _ in chi1_groups}
for ts in u.trajectory:
    for name, ag in chi1_groups:
        angles[name].append(ag.dihedral.value() % 360)

fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(0, 360, 73)
for name, vals in angles.items():
    ax.hist(vals, bins=bins, density=True, histtype="step", linewidth=1.8, label=name)
for b in (120, 240):
    ax.axvline(b, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("χ1 (degrees)")
ax.set_ylabel("Probability density")
ax.set_xlim(0, 360)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

There are three peaks, with valleys near **120°** and **240°** (dashed lines). So we use three states, called X, Y and Z, with boundaries `[120, 240]`.

For nucleic acids, make the same plot for the sugar-base dihedral and set `No_of_peaks_nucleic`, `peak_boundaries_nucleic` and `states_nucleic` in the same way. The defaults (`[140, 330]`, `"SAS"`) describe syn / anti / syn.

## 4. Compute the conditional activity

`mean_conditional_activity()` runs the whole calculation and returns three things: the conditional activity matrix, the persistence time of each residue, and the exchange-time matrix.

A residue needs **at least 10 transitions** to be included. Residues with fewer get `NaN`, and CONDACT prints a message for each affected pair. That is expected for this short example (one residue rarely switches state), and it is why long trajectories are needed for real systems.

In [ ]:
study = CONDACT(
    universe=u,
    selected_resid=SELECTED_RESID,
    No_of_peaks_protein=3,               # number of chi1 states seen in the plot above
    peak_boundaries_protein=[120, 240],  # the valleys between them
    states_protein="XYZ",                # one letter per state
    No_of_peaks_nucleic=3,               # only used if nucleotides are selected
    peak_boundaries_nucleic=[140, 330],
    states_nucleic="SAS",
    saving_frequency=SAVING_FREQUENCY,
    keep_negative=False,                 # False = use 0-360 degrees
)

conditional_activity_list, persistence_times, exchange_times = study.mean_conditional_activity()

CA = np.array(conditional_activity_list, dtype=float)
_, residue_names, residue_type = study.residues()
labels = [f"{n} {r.resid}" for n, r in zip(residue_names, residue_type)]

The run also writes these files to the working directory (click the folder icon on the left to see and download them):

| File | Contents |
|---|---|
| `Dihedral_Angle.csv` | the dihedral angle of every selected residue at every frame |
| `transition_times.pkl` | the times at which each residue changed state |
| `wait_times.pkl` | the waiting times between transitions |
| `Conditional_Activity_matrix.npy` | the N × N conditional activity matrix |
| `Dynamic_Memory.npy` | its diagonal |

## 5. Look at the results

### Transitions and persistence times

In [ ]:
summary = pd.DataFrame({
    "transitions": [len(study.transition_times[l]) for l in labels],
    "persistence time (ps)": [persistence_times[l] for l in labels],
    "dynamical memory A[X][X]": np.diag(CA),
}, index=labels)
summary.round(3)

**Dynamical memory** is the diagonal of the matrix. A positive value means the residue's transitions are bunched in time rather than random: once it switches, it tends to switch again soon.

### The conditional activity matrix

Read each cell as **row X responding to column Y**: a positive value means a transition in the column residue is followed sooner than expected by a transition in the row residue.

In [ ]:
display(pd.DataFrame(CA, index=labels, columns=labels).round(3))

fig, ax = plt.subplots(figsize=(4.8, 4))
im = ax.imshow(CA, cmap="hot_r", vmin=0, origin="upper")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.set_xlabel("Y (residue that transitions first)")
ax.set_ylabel("X (residue that responds)")
fig.colorbar(im, label="Conditional activity A[X][Y]")
plt.tight_layout()
plt.show()

### Dynamically connected domains (optional)

The principal eigenvector of the symmetrized matrix groups residues that are kinetically coupled to each other. With three residues this is only a demonstration; on a real protein, the residues with the largest components form the dominant dynamically connected domain.

In [ ]:
CA_filled = np.nan_to_num(CA, nan=0.0)
CA_sym = 0.5 * (CA_filled + CA_filled.T)
eigenvalues, eigenvectors = np.linalg.eigh(CA_sym)
principal = eigenvectors[:, np.argmax(eigenvalues)]
principal = principal * np.sign(principal[np.argmax(np.abs(principal))])  # fix the overall sign

print(f"Largest eigenvalue: {eigenvalues.max():.3f}")
pd.Series(principal, index=labels, name="principal eigenvector").round(3)

## 6. Which couplings are statistically significant?

`CONDACT_FDR` tests every off-diagonal entry with a permutation test: the transition times of Y are shifted by a random amount (wrapping around the trajectory) to break any timing relationship with X, and the conditional activity is recomputed. The p-value is the fraction of shifts that give a value at least as large as the observed one. P-values are then corrected for multiple testing with Benjamini–Hochberg FDR.

`n_perm` controls the number of random shifts. 2,000 keeps this example fast; use more (for example 10,000 or above) for publication-quality p-values.

In [ ]:
fdr = CONDACT_FDR(
    transition_times_file="transition_times.pkl",
    CA_matrix_file="Conditional_Activity_matrix.npy",
    total_time=TOTAL_TIME,   # length of the simulation in ps
    n_perm=2000,
    q=0.05,                  # FDR threshold
    CA_cutoff=0.0,           # only test entries with CA >= this value
    random_seed=0,
)
pvals, qvals, significant = fdr.run()

pd.read_csv("CA_all_tested_edges.csv")

`Residue_i` is the responding residue X (row) and `Residue_j` is the residue Y that transitions first (column). The same table is saved as `CA_all_tested_edges.csv`, and the significant rows alone as `CA_significant_edges.csv`.

## 7. Trace a communication route

`CONDACT_CommPath` combines the conditional activity matrix with the average distance between residues. It builds a directed graph with an edge **Y → X** wherever `A[X][Y]` is above `min_ca` and the two residues are closer than `cutoff` (Å), then follows the strongest edges.

First, the average residue-residue distance matrix:

In [ ]:
from MDAnalysis.analysis import distances

residues_sel = u.select_atoms(f"resid {SELECTED_RESID} and not (resname ALA or resname GLY)")
dist = np.zeros((len(labels), len(labels)))
for ts in u.trajectory:
    centers = residues_sel.center_of_geometry(compound="residues")
    dist += distances.distance_array(centers, centers, box=u.dimensions)
dist /= len(u.trajectory)

pd.DataFrame(dist, index=labels, columns=labels).round(1)

In [ ]:
comm = CONDACT_CommPath(CA_filled, dist, labels=labels)

# Start from the source of the strongest edge and list the best-scoring paths.
# The 30 A cutoff is deliberately loose for this tiny example; for a real system
# use a contact-like distance (for example 5-8 A).
paths = comm.enumerate_paths(start=None, start_mode="global_max",
                             cutoff=30.0, min_ca=0.0, max_depth=5, top_k=10)
comm.print_paths(paths, top=10)

To ask about a specific pair, give residue labels, for example:

```python
paths = comm.trace_greedy(start="ASP 3", target="GLN 1", cutoff=30.0, min_ca=0.0, branch_k=10)
comm.print_paths(paths)
```

`branch_k=1` follows only the single strongest edge at each step; larger values keep several candidate paths (beam search).

## 8. Use your own trajectory

1. Get your files into Colab, either by uploading them or by mounting Google Drive:

```python
from google.colab import files
uploaded = files.upload()          # choose your topology and trajectory
```

```python
from google.colab import drive
drive.mount("/content/drive")      # files are then under /content/drive/MyDrive/...
```

2. In section 2, point `topology_file` and `trajectory_file` at your files, and set `SELECTED_RESID` and `SAVING_FREQUENCY` for your system.
3. Re-run the notebook from section 2. Redo the plot in section 3 and update the boundaries in section 4 if your peaks sit in different places.

**Things to keep in mind**

- Residues need at least 10 transitions, so use long trajectories that sample many side-chain flips. Sparse transitions give `NaN` or unstable values.
- The cost grows with the square of the number of selected residues. Start with a region of interest before running a whole complex.
- Splitting a continuous angle into a few states is a coarse-graining; check the distribution rather than relying on the defaults.
- Colab's free runtime has limited memory and disconnects when idle. For large systems, strip solvent and subsample the trajectory first, or run locally.
- The trajectories from the paper are on Zenodo: https://zenodo.org/records/17193099

## Citation

If you use CONDACT, please cite:

- Onyema, A. C.; Dikeocha, C.; Patel, R.; Moussa, J.; Loverde, S. M. *Mapping Allosteric Communication in the Nucleosome with Conditional Activity.* J. Chem. Inf. Model. (2026). https://pubs.acs.org/doi/10.1021/acs.jcim.6c00317
- Lin, M. M. *Timing correlations in proteins predict functional modules and dynamic allostery.* J. Am. Chem. Soc. 2016, 138 (15), 5036–5043. https://pubs.acs.org/doi/10.1021/jacs.5b08814
- MDAnalysis: https://www.mdanalysis.org/pages/citations/

Code: https://github.com/CUNY-CSI-Loverde-Laboratory/conditional_activity (GPL v2)